# Fold-1 Segmentation Engine — Small Dataset Trial

Self-contained smoke-test run. See `TASK_09_segmentation_engine.md` for the full spec.

**Before running:** set the runtime to GPU (Runtime → Change runtime type → T4 GPU),
and upload `fold1_dataset.zip` to your Drive (`MyDrive/` root). The setup cell
clones the code and stages the data automatically. This validates the training
loop, not model quality — 33 tiles is a trial set.

In [ ]:
!pip install -q segmentation-models-pytorch albumentations rasterio pyyaml


In [ ]:
# --- Setup: code + data ---------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# Code: clone the task branch (always in sync with PR #1)
!git clone -q -b seg-engine-v1 https://github.com/HeissenBergei/QGIS-aPJ /content/QGIS-aPJ
%cd /content/QGIS-aPJ/tasks/09_segmentation_engine

# Data: upload fold1_dataset.zip to your Drive (MyDrive root by default),
# then stage it as ./data (config.yaml expects data.root == "data").
DATA_ZIP = '/content/drive/MyDrive/fold1_dataset.zip'   # <-- adjust if elsewhere
!rm -rf data /content/_ds
!unzip -q "{DATA_ZIP}" -d /content/_ds && mv /content/_ds/fold1_dataset data
!echo "train entries:" $(wc -l < data/splits/train.txt) "| val entries:" $(wc -l < data/splits/val.txt)

# Confirm a GPU is attached: Runtime -> Change runtime type -> T4 GPU
import torch
print("CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - fix runtime")

In [ ]:
# Sanity check: confirm data contract before burning a training run on
# a malformed dataset
import os
assert os.path.exists('data/images'), 'data/images missing'
assert os.path.exists('data/masks'), 'data/masks missing'
assert os.path.exists('data/splits/train.txt'), 'train split missing'
assert os.path.exists('data/splits/val.txt'), 'val split missing'

import numpy as np
import rasterio
from dataset import ADDON_INDEX
sample_id = open('data/splits/train.txt').readline().strip()
# images/  = addon "<stem>_satellite.png" (RGB); masks/ = "<stem>_mask_index.png"
with rasterio.open(f'data/images/{sample_id}.png') as src:
    print('image shape:', src.read().shape, 'dtype:', src.dtypes)
with rasterio.open(f'data/masks/{sample_id}.png') as src:
    idx = src.read(1)  # single-channel class-index map (multi-class, not 4-band)
    print('mask (index map) shape:', idx.shape, 'dtype:', idx.dtype)
    print('unique class indices present:', np.unique(idx))
    print('Task 09 channels <- addon index:', ADDON_INDEX)
    print('dataset.py expands this single-channel index map into 4 binary channels')

In [ ]:
# For the initial trial: shrink epochs and batch size in config.yaml
# (or override here) to get a fast smoke-test run before a full pass.
import yaml
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['train']['epochs'] = 10
cfg['train']['batch_size'] = 4
with open('config_trial.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
print('Wrote config_trial.yaml for smoke test')


In [ ]:
!python train.py --config config_trial.yaml


In [ ]:
# Persist outputs — /content is wiped when the runtime ends.
!mkdir -p /content/drive/MyDrive/zoning_seg_engine_out
!cp -v checkpoints/best_model.pt /content/drive/MyDrive/zoning_seg_engine_out/ \
    || echo "no checkpoint written (did training reach a best epoch?)"
print("Saved to MyDrive/zoning_seg_engine_out/")

## Reading the results
Check per-class IoU independently — do not judge the run on the mean alone.
`parcel_border` may lag the other three; before treating it as a bug, pull
up a few failing validation tiles and check whether the boundary is
actually visible in the imagery (see TASK_09 'known hard case').